# 03 · Forecast Jun–Dec 2025

Revenue = passengers x average fare, forecast separately because they behave differently:
- **Passengers** grow steadily with capacity. Model: log-linear trend plus a seasonal index from 2024, backtested on Mar–May 2025.
- **Fares** stepped up ~39% in Nov 2024 (a structural break). The forecast uses the post-break level, with two scenarios.

Bands come from 2,000 bootstrap draws of the model's monthly residuals.

In [ ]:
import sys, json, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from src.metrics import load_clean, headline, DATA_DIR, OUTPUT_DIR, DATA_END
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
NAVY, VIOLET, LILAC, ORANGE, SLATE, GREEN = "#030734", "#682EFF", "#CD83FF", "#F05100", "#62748E", "#009588"
sns.set_theme(style="whitegrid", palette=[VIOLET, LILAC, NAVY, ORANGE, SLATE, GREEN])
FIG = OUTPUT_DIR / "figures"; FIG.mkdir(parents=True, exist_ok=True); DATA_DIR.mkdir(parents=True, exist_ok=True)
Cr = lambda v: f"₹{v/1e7:,.2f} Cr"; Lk = lambda v: f"₹{v/1e5:,.1f} L"
findings = []
def note(q, topic, text):
    findings.append(dict(q=q, topic=topic, finding=text)); print(f"[{q}] {topic}: {text}")
def save(fig, name):
    fig.savefig(FIG / f"{name}.png", dpi=150, bbox_inches="tight"); plt.show()
def kpis(g):
    return pd.Series({"flights": len(g), "revenue": g.Revenue.sum(), "pax": g.PaxCount.sum(), "seats": g.Capacity.sum(),
                      "ask": g.ask.sum(), "rpk": g.rpk.sum(), "lf": g.PaxCount.sum() / g.Capacity.sum(),
                      "fare": g.Revenue.sum() / g.PaxCount.sum(), "rask": g.Revenue.sum() / g.ask.sum(),
                      "yield_rpk": g.Revenue.sum() / g.rpk.sum(), "stage_km": g.ask.sum() / g.Capacity.sum()})

In [ ]:
f = pd.read_parquet(DATA_DIR / "flights.parquet")
NET = kpis(f)

In [ ]:
Mo = f.groupby("month").apply(kpis, include_groups=False); Mo.index = pd.to_datetime(Mo.index)
t = np.arange(len(Mo)); moy = Mo.index.month
def fit_pax(idx):
    """log(pax) = a + b*t + seasonal(month), seasonal estimated from 2024 detrended residuals (month dummies, ridge-shrunk)."""
    y = np.log(Mo.pax.values[idx]); tt = t[idx]; b, a = np.polyfit(tt, y, 1)
    res = pd.Series(y - (a + b * tt), index=moy[idx]).groupby(level=0).mean() * 0.5   # shrink: one year of seasonality only
    return a, b, res
def pred_pax(params, tt, months):
    a, b, s = params; return np.exp(a + b * tt + np.array([s.get(m, 0.0) for m in months]))
train = np.arange(len(Mo) - 3); test = np.arange(len(Mo) - 3, len(Mo))
pbt = fit_pax(train); bt_pred = pred_pax(pbt, t[test], moy[test])
naive = np.repeat(Mo.pax.values[train][-3:].mean(), 3)
bt = pd.DataFrame({"actual": Mo.pax.values[test], "model": bt_pred, "naive_last3": naive}, index=Mo.index[test])
bt_err = bt.model.sum() / bt.actual.sum() - 1; nv_err = bt.naive_last3.sum() / bt.actual.sum() - 1
display(bt); bt.to_csv(OUTPUT_DIR / "forecast_backtest.csv")
note("Backtest", "Passengers Mar-May 2025", f"Trend + seasonal model {bt_err:+.1%} on the 3-month total vs {nv_err:+.1%} for a naive 'last 3 months' forecast.")

In [ ]:
P = fit_pax(np.arange(len(Mo)))
fc_idx = pd.date_range("2025-06-01", "2025-12-01", freq="MS"); tf = np.arange(len(Mo), len(Mo) + len(fc_idx))
pax_fc = pred_pax(P, tf, fc_idx.month)
fit_in = pred_pax(P, t, moy); resid = np.log(Mo.pax.values) - np.log(fit_in)
post = Mo.loc["2024-11-01":]; fare_hi = post.revenue.sum() / post.pax.sum()
recent = Mo.loc["2025-03-01":]; fare_lo = recent.revenue.sum() / recent.pax.sum()
fare_resid = np.log(post.fare / fare_hi).values
rng = np.random.default_rng(42); N = 2000
sims = {}
for name, level in {"A · Fares hold (post-Nov 2024 level)": fare_hi, "B · Fares settle (Mar-May 2025 level)": fare_lo}.items():
    pe = np.exp(rng.choice(resid, (N, len(fc_idx)))); fe = np.exp(rng.choice(fare_resid, (N, len(fc_idx))))
    pax = pax_fc * pe; rev = pax * level * fe
    sims[name] = {"pax": pax, "revenue": rev}
rows = []
for name, s in sims.items():
    for metric, arr in s.items():
        for i, m in enumerate(fc_idx):
            rows.append(dict(scenario=name, month=m, metric={"pax": "Passengers", "revenue": "Revenue"}[metric], kind="Forecast",
                             actual=None, p10=np.percentile(arr[:, i], 10), p50=np.percentile(arr[:, i], 50), p90=np.percentile(arr[:, i], 90)))
    for metric, col in (("Passengers", "pax"), ("Revenue", "revenue")):
        for m, v in Mo[col].items():
            rows.append(dict(scenario=name, month=m, metric=metric, kind="Actual", actual=v, p10=None, p50=None, p90=None))
FC = pd.DataFrame(rows); FC.to_csv(OUTPUT_DIR / "pbi_forecast.csv", index=False, date_format="%Y-%m-%d")
tot = {name: {k: np.percentile(v.sum(axis=1), [10, 50, 90]) for k, v in s.items()} for name, s in sims.items()}
ann = []
ytd25 = f[f.date.dt.year == 2025]
for name, d in tot.items():
    for k, nice in (("pax", "Passengers"), ("revenue", "Revenue")):
        base = ytd25.PaxCount.sum() if k == "pax" else ytd25.Revenue.sum()
        ann.append(dict(scenario=name, metric=f"{nice} 2025 full year", p10=base + d[k][0], p50=base + d[k][1], p90=base + d[k][2]))
        ann.append(dict(scenario=name, metric=f"{nice} Jun-Dec 2025", p10=d[k][0], p50=d[k][1], p90=d[k][2]))
FA = pd.DataFrame(ann); FA.to_csv(OUTPUT_DIR / "pbi_forecast_2025.csv", index=False); display(FA)
rev24 = f[f.date.dt.year == 2024].Revenue.sum()
a = FA[(FA.metric == "Revenue 2025 full year") & FA.scenario.str.startswith("A")].iloc[0]
b = FA[(FA.metric == "Revenue 2025 full year") & FA.scenario.str.startswith("B")].iloc[0]
note("Forecast", "2025 revenue", f"Full-year 2025 revenue {Cr(a.p50)} if fares hold (80% range {Cr(a.p10)}-{Cr(a.p90)}), {Cr(b.p50)} if they settle at the Mar-May level, "
     f"vs {Cr(rev24)} in 2024 ({a.p50/rev24-1:+.0%} / {b.p50/rev24-1:+.0%}).")
pa = FA[(FA.metric == "Passengers 2025 full year") & FA.scenario.str.startswith("A")].iloc[0]
note("Forecast", "2025 passengers", f"About {pa.p50/1e6:.2f} M passengers in 2025 ({pa.p50/f[f.date.dt.year == 2024].PaxCount.sum()-1:+.0%} vs 2024). "
     "The summer months (Jul-Oct) are the weakest in the seasonal pattern.")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.plot(Mo.index, Mo.revenue / 1e7, color=NAVY, marker="o", label="Actual")
for (name, s), c in zip(sims.items(), [VIOLET, ORANGE]):
    q = np.percentile(s["revenue"], [10, 50, 90], axis=0) / 1e7
    ax.plot(fc_idx, q[1], color=c, lw=2, label=name); ax.fill_between(fc_idx, q[0], q[2], color=c, alpha=.15)
ax.set_ylabel("₹ crore"); ax.legend(fontsize=8); ax.set_title("Monthly revenue: actual and forecast (P10-P90)"); save(fig, "f01_revenue_forecast")
F = pd.DataFrame(findings); F.to_csv(OUTPUT_DIR / "forecast_findings.csv", index=False); F